# Train the mobility-aid detector on Kaggle

Runs the full training on a free Kaggle GPU and saves `aids_best.pt` for download.

**Before running**
1. Settings (right panel): Accelerator **GPU T4 x2** or **P100**, Internet **On**.
2. Add-ons → Secrets: add `GITHUB_TOKEN` (a GitHub token with read access to this private repo) and `ROBOFLOW_API_KEY`.
3. Run all cells. Training 100 epochs on YOLO11s takes roughly 1–3 hours depending on dataset size.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["ROBOFLOW_API_KEY"] = secrets.get_secret("ROBOFLOW_API_KEY")
token = secrets.get_secret("GITHUB_TOKEN")

In [ ]:
!git clone https://{token}@github.com/mabutalib1/smart-metro-assistant.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

## 1. Download and merge the datasets
Check the printed class counts and the list of dropped classes. If a class you need was dropped, add it to `CLASS_MAP` in `data/merge_datasets.py`.

In [ ]:
!python data/merge_datasets.py

## 2. (Optional) add your own images
Upload a Roboflow export of your own labeled images (YOLOv8 format) as a Kaggle dataset, then copy it into `data/raw/` and merge again.

In [ ]:
# Example, adjust the input path to your uploaded dataset:
# !cp -r /kaggle/input/our-cane-images data/raw/own_images
# !python data/merge_datasets.py --skip-download

## 3. Train

In [ ]:
!python train/train_detector.py --model yolo11s.pt --epochs 50 --batch 32 --device 0

## 4. Check the results

In [ ]:
from IPython.display import Image, display
for name in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
    path = f"runs/aids/{name}"
    if os.path.exists(path):
        display(Image(filename=path, width=900))

## 5. Save the weights
Download `aids_best.pt` from the Output tab, put it in `models/`, and set `models.aids: models/aids_best.pt` in `configs/config.yaml`.

In [ ]:
import shutil
shutil.copy("models/aids_best.pt", "/kaggle/working/aids_best.pt")
print("Saved /kaggle/working/aids_best.pt")